# QMIX: Value Decomposition in Multi-Agent RL

This notebook demonstrates the QMIX algorithm - a value decomposition method for multi-agent reinforcement learning.

## What is QMIX?

QMIX learns a **monotonic** value function factorization:
- Each agent has an individual Q-network: Q_i(obs_i, a_i)
- A mixing network combines them: Q_tot = f(Q_1, Q_2, ..., Q_n, global_state)
- **Monotonicity constraint**: ∂Q_tot/∂Q_i ≥ 0 for all agents

This ensures that if an agent improves its individual Q-value, the global Q-value also improves!

## Key Differences from MADDPG

| Feature | MADDPG | QMIX |
|---------|--------|------|
| Action Space | Continuous | **Discrete** |
| Architecture | Actor-Critic | **Value-based (Q-learning)** |
| Credit Assignment | Centralized critic | **Value decomposition** |
| Exploration | OU noise | **Epsilon-greedy** |

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath(os.getcwd())))

import numpy as np
import torch
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## Setup Environment

For QMIX, we need to adapt our continuous action environment to discrete actions.

In [ ]:
from environments.cooperative_navigation import CooperativeNavigation

# For QMIX, we discretize actions
class DiscreteCoopNav(CooperativeNavigation):
    """Cooperative Navigation with discrete actions"""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # 5 discrete actions: up, down, left, right, stay
        self.action_dim = 5
        self.discrete_actions = np.array([
            [0, 0.5],   # up
            [0, -0.5],  # down
            [-0.5, 0],  # left
            [0.5, 0],   # right
            [0, 0]      # stay
        ])
    
    def step(self, discrete_actions):
        """Convert discrete actions to continuous"""
        continuous_actions = [self.discrete_actions[a] for a in discrete_actions]
        return super().step(continuous_actions)

# Create environment
env = DiscreteCoopNav(num_agents=3, num_landmarks=3)
env_info = env.get_env_info()
env_info['action_dim'] = 5  # Override with discrete actions

print(f"Environment Info:")
print(f"  Agents: {env_info['num_agents']}")
print(f"  State dim: {env_info['state_dim']}")
print(f"  Action dim: {env_info['action_dim']} (discrete)")

## Initialize QMIX Agent

In [ ]:
from algorithms.qmix import QMIX

agent = QMIX(
    num_agents=env_info['num_agents'],
    state_dim=env_info['state_dim'],
    action_dim=5,  # Discrete actions
    hidden_dim=64,
    mixing_embed_dim=32,
    lr=5e-4,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay=0.995,
    batch_size=64,
    device='cpu'
)

print("QMIX agent created!")
print(f"  Number of Q-networks: {len(agent.q_networks)}")
print(f"  Mixing network: {agent.mixing_network}")
print(f"  Initial epsilon: {agent.epsilon}")

## Visualize QMIX Architecture

In [ ]:
from IPython.display import Image, display

# Create architecture diagram
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.axis('off')

# Draw components
y_positions = [0.8, 0.5, 0.2]
colors = ['lightblue', 'lightgreen', 'lightcoral']

# Q-Networks
for i, (y, color) in enumerate(zip(y_positions, colors)):
    rect = plt.Rectangle((0.1, y-0.05), 0.2, 0.1, 
                         facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.2, y, f'Q-Net {i}\n(obs_{i} → Q_{i})', 
           ha='center', va='center', fontsize=10, weight='bold')

# Mixing Network
rect = plt.Rectangle((0.5, 0.35), 0.3, 0.3,
                     facecolor='yellow', edgecolor='black', linewidth=2)
ax.add_patch(rect)
ax.text(0.65, 0.5, 'Mixing Network\n(Q₁, Q₂, Q₃, s → Q_tot)',
       ha='center', va='center', fontsize=11, weight='bold')

# Arrows
for y in y_positions:
    ax.annotate('', xy=(0.5, 0.5), xytext=(0.3, y),
               arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('QMIX Architecture', fontsize=14, weight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\nKey Insight: The mixing network ensures monotonicity!")
print("If any agent improves its Q-value, Q_tot improves too.")

## Training Loop

In [ ]:
# Training
num_episodes = 300
max_steps = 100

episode_rewards = []
epsilons = []
losses = []

for episode in range(num_episodes):
    states = env.reset()
    episode_reward = 0
    episode_loss = []
    
    for step in range(max_steps):
        # Select actions with epsilon-greedy
        actions = agent.act(states)
        
        # Execute
        global_state = env.get_global_state()
        next_states, rewards, dones, info = env.step(actions)
        next_global_state = env.get_global_state()
        
        # Store
        agent.step(states, actions, rewards, next_states, dones,
                  global_state, next_global_state)
        
        # Update
        if len(agent.replay_buffer) > agent.batch_size:
            update_info = agent.update()
            if update_info:
                episode_loss.append(update_info['loss'])
        
        states = next_states
        episode_reward += sum(rewards)
        
        if all(dones):
            break
    
    episode_rewards.append(episode_reward)
    epsilons.append(agent.epsilon)
    if episode_loss:
        losses.append(np.mean(episode_loss))
    
    if (episode + 1) % 30 == 0:
        avg_reward = np.mean(episode_rewards[-30:])
        print(f"Episode {episode + 1}: Avg Reward={avg_reward:.2f}, ε={agent.epsilon:.3f}")

print("\nTraining completed!")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Rewards
axes[0].plot(episode_rewards, alpha=0.3, label='Raw')
window = 30
smoothed = [np.mean(episode_rewards[max(0, i-window):i+1]) for i in range(len(episode_rewards))]
axes[0].plot(smoothed, linewidth=2, label='Smoothed')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('QMIX Training Rewards')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Epsilon decay
axes[1].plot(epsilons, linewidth=2, color='orange')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Epsilon')
axes[1].set_title('Exploration Rate (ε-greedy)')
axes[1].grid(alpha=0.3)

# Loss
if losses:
    axes[2].plot(losses, alpha=0.6, color='red')
    axes[2].set_xlabel('Update')
    axes[2].set_ylabel('TD Loss')
    axes[2].set_title('Training Loss')
    axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Test Trained Agent

In [ ]:
# Test with greedy policy (epsilon=0)
test_rewards = []

for episode in range(5):
    states = env.reset()
    episode_reward = 0
    
    for step in range(max_steps):
        actions = agent.act(states, epsilon=0.0)  # Greedy
        states, rewards, dones, info = env.step(actions)
        episode_reward += sum(rewards)
        
        if all(dones):
            break
    
    test_rewards.append(episode_reward)
    print(f"Test Episode {episode + 1}: Reward = {episode_reward:.2f}")

print(f"\nTest Performance: {np.mean(test_rewards):.2f} ± {np.std(test_rewards):.2f}")

## Understanding the Mixing Network

Let's visualize how the mixing network combines individual Q-values.

In [ ]:
# Get some sample data
states = env.reset()
global_state = env.get_global_state()

# Get individual Q-values
with torch.no_grad():
    q_values_list = []
    for i, state in enumerate(states):
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        q_vals = agent.q_networks[i](state_tensor)
        q_values_list.append(q_vals.numpy()[0])

# Visualize Q-values for each agent
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
action_names = ['Up', 'Down', 'Left', 'Right', 'Stay']

for i, (ax, q_vals) in enumerate(zip(axes, q_values_list)):
    bars = ax.bar(action_names, q_vals)
    max_idx = np.argmax(q_vals)
    bars[max_idx].set_color('red')
    
    ax.set_title(f'Agent {i} Q-values', fontweight='bold')
    ax.set_ylabel('Q-value')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nThe mixing network combines these individual Q-values")
print("while ensuring monotonicity (each agent's improvement helps the team).")

## Key Takeaways

1. **Value Decomposition**: QMIX decomposes global Q-value into individual Q-values
2. **Monotonicity**: Ensures individual improvements benefit the team
3. **Discrete Actions**: Works with discrete action spaces (unlike MADDPG)
4. **Credit Assignment**: Better credit assignment than independent Q-learning

## When to Use QMIX?

✅ **Use QMIX when:**
- You have discrete actions
- You need explicit value decomposition
- Credit assignment is challenging

❌ **Use MADDPG when:**
- You have continuous actions
- You prefer policy gradient methods

## Next Steps

- Compare QMIX with MADDPG and CommMADDPG
- Try on different environments
- Tune hyperparameters (mixing_embed_dim, learning rate, etc.)
- Explore advanced variants (QPLEX, QTRAN)